# DATA CLEANING AND CREATE FEATURES

## SETUP AND IMPORT

In [1]:
import duckdb
import pandas as pd
import sqlite3
from pathlib import Path
import holidays
import os
import numpy as np
from functions import filter_train_type, historic_delay_features, create_features

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_with_weather.parquet')
            """)

df_raw = con.execute("SELECT * FROM train_delay").fetchdf()

## DATA INSPECTION

In [2]:
print(df_raw.info())                      
# print(df_raw.head(5))                      
# print(df_raw["delay_in_min"].describe())  
# print(df_raw["train_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2175996 entries, 0 to 2175995
Data columns (total 15 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   canceled           bool          
 6   arrival_planned    datetime64[us]
 7   arrival_real       datetime64[us]
 8   departure_planned  datetime64[us]
 9   departure_real     datetime64[us]
 10  delay              int32         
 11  temperature        float64       
 12  precipitation      float64       
 13  wind_speed         float64       
 14  weather_status     object        
dtypes: bool(1), datetime64[us](4), float64(3), int32(1), int64(1), object(5)
memory usage: 226.2+ MB
None


## DATA CLEANING

In [3]:
### FILTER TRAIN TYPES (ONLY ICE AND IC)
df_cleaned = filter_train_type(df_raw)

### REMOVE RIDES THAT WERE CANCELED
df_cleaned = (
    df_cleaned.groupby("ride_id")
      .filter(lambda g: not g["canceled"].any())
)

### REMOVE UNECESSARY COLUMNS
df_cleaned = df_cleaned.drop(columns=["weather_status", "canceled"])

### RENAME COLUMNS
df_cleaned = df_cleaned.rename(columns={
    "station_name": "station_current",
    "final_destination_station": "station_dest"})

# REARRANGE COLUMNS
df_cleaned = df_cleaned.loc[:, ["ride_id", "train_name", "train_type", "station_current", "station_dest",
               "departure_planned", "departure_real", "arrival_planned", "arrival_real",
                "temperature", "precipitation", "wind_speed", "delay"]]


print(df_cleaned.columns.tolist())
print(df_cleaned.info())    


['ride_id', 'train_name', 'train_type', 'station_current', 'station_dest', 'departure_planned', 'departure_real', 'arrival_planned', 'arrival_real', 'temperature', 'precipitation', 'wind_speed', 'delay']
<class 'pandas.core.frame.DataFrame'>
Index: 1842266 entries, 0 to 2175995
Data columns (total 13 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   departure_planned  datetime64[us]
 6   departure_real     datetime64[us]
 7   arrival_planned    datetime64[us]
 8   arrival_real       datetime64[us]
 9   temperature        float64       
 10  precipitation      float64       
 11  wind_speed         float64       
 12  delay              int32         
dtypes: datetime64[us](4), float64(3), int32(1), int64(1), object(4)
memory usage: 189.7+ MB
None


## NA ANALYSIS

In [4]:
# number of na per column
print(df_cleaned.isnull().sum(axis = 0))

# missings in time_current_departure_planned and time_current_arrival_plannned
# are due to first or last station that do not have either arrival time (start station) or departure time (destination station)
len(df_cleaned["ride_id"].unique())

ride_id                   0
train_name                0
train_type                0
station_current           0
station_dest              0
departure_planned    200154
departure_real       200031
arrival_planned      200154
arrival_real         200036
temperature               0
precipitation             0
wind_speed                0
delay                     0
dtype: int64


200154

## CREATE FEATURES

### STATION START

In [4]:
### REASON: ALSO EXISTS IN API DATA
### AFTER THAT, WE CAN APPLY FUNCTIONS TO BOTH DATASETS

# sorting
df_cleaned.sort_values(by=["ride_id", "departure_planned"])

# grouping
grouped = df_cleaned.groupby("ride_id")

# create station_start
df_cleaned["station_start"] = grouped["station_current"].transform("first")

### OTHER FEATURES (WITH FUNCTION)

In [ ]:
# extract the historical features first
hist_df = historic_delay_features(df_cleaned)

In [ ]:
# get the information for the lookup file
max_date = df_cleaned["departure_real"].max().floor("D")

# last 60 days
window_start = max_date - pd.Timedelta("60D")

# select last 60 days
df_last60 = df_cleaned[
    (df_cleaned["departure_real"] > window_start) &
    (df_cleaned["departure_real"] <= max_date)
]

# aggregate data
hist_delay_lookup = df_last60.groupby("station_current").agg(
    hist_delay_avg = ("delay", "mean"),
    hist_delay_q90 = ("delay", lambda x: x.quantile(0.9)),
    hist_delay_count = ("delay", "count")
).reset_index()

In [5]:
max_date = df_cleaned["departure_real"].max().floor("D")
print(max_date)

2025-12-31 00:00:00


In [ ]:
# df_features = create_features(df_cleaned, True)

NameError: name 'hist_df' is not defined

## TARGET TRANSFORMATION

In [ ]:
df_features["delay_cat"] = pd.cut(
    df_features["delay"],
    bins=[-np.inf, 60, 120, np.inf],
    labels=[0, 1, 2],
)

df_features["delay_cat"].value_counts(normalize = True)

delay_cat
0    0.980727
1    0.017127
2    0.002146
Name: proportion, dtype: float64

## GENERATE HISTORICAL DELAY LOOKUP FILE

## EXPORT DATA

In [ ]:
df_features.to_parquet(
    'data/train_delay_cleaned.parquet', 
    compression='zstd', 
    index=False
)